# MRIscanner training (Colab GPU)

Runs the glioma-recall fix (heavier augmentation, glioma-weighted loss, no label
smoothing, plus a direct penalty on false "no tumor" predictions) on a free Colab GPU
instead of a loaded local Mac.

### One-time setup before you hit Run All
1. Runtime -> Change runtime type -> **T4 GPU**.
2. Get your `kaggle.json`: kaggle.com -> Settings -> API -> **Create New Token** -- this
   downloads an actual `kaggle.json` file (not the copyable token string; that path goes
   through a Kaggle endpoint that currently 404s on the published `kaggle` package).
3. Click the **folder icon** in the left sidebar and drag that `kaggle.json` into the
   file list at `/content/`. A plain file drop in your own browser -- nobody else sees it.
4. **Runtime -> Run all.** Everything after that (clone, patch, install, download the
   real dataset, train all 3 architectures, benchmark, zip + download the results) runs
   unattended -- no further clicks needed.


In [ ]:
!nvidia-smi


In [ ]:
import os, subprocess

REPO_DIR = "/content/MRIscanner"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(
        ["git", "clone", "https://github.com/christiandrep7/MRIscanner.git", REPO_DIR],
        check=True,
    )
os.chdir(REPO_DIR)
assert os.path.isdir("src"), f"expected src/ under {os.getcwd()} -- clone did not land where expected"
print("cwd:", os.getcwd())


### Patch in this session's fix (config.py, data.py, train.py)
Not pushed to `main` yet -- these 3 cells overwrite the freshly-cloned copies with the local fix.
(Each cell below starts by re-asserting the working directory, so re-running cells out of order -- or a full Run All after a previous partial run -- can't silently write into the wrong place.)

In [ ]:
%%writefile src/config.py
from dataclasses import dataclass, field
from pathlib import Path


@dataclass
class DataConfig:
    data_root: Path = Path("data")
    train_dir_name: str = "Training"
    test_dir_name: str = "Testing"
    image_size: int = 224
    batch_size: int = 32
    num_workers: int = 2
    val_split: float = 0.15
    seed: int = 42


@dataclass
class TrainConfig:
    architecture: str = "resnet50"
    epochs: int = 15
    learning_rate: float = 1e-4
    weight_decay: float = 1e-4
    # Label smoothing softened every class's decision, not just the weak one --
    # measured worse: overall accuracy and meningioma recall both dropped. Left
    # here at 0.0 (off) so it stays available without being applied by default.
    label_smoothing: float = 0.0
    # Per-class CrossEntropyLoss weight, keyed by class *name* (resolved against
    # the dataset's own class_to_idx at train time, so it doesn't depend on
    # alphabetical class order). glioma benchmarked with the lowest recall of all
    # 4 classes on data/Testing across all 3 architectures (~76-84% vs 90-100%
    # elsewhere) -- weighting it up makes the loss penalize a missed glioma more
    # than a missed meningioma/notumor/pituitary, pushing recall up directly
    # instead of via a blanket recipe change like label smoothing.
    class_loss_weights: dict = field(default_factory=lambda: {"glioma": 1.5})
    # class_loss_weights pushed overall glioma recall up, but empirically didn't
    # target *which* wrong class it slips into -- glioma-called-"notumor" (the
    # most dangerous miss: an "all clear" on a real tumor) stayed flat/got slightly
    # worse. This adds a direct penalty on the softmax probability mass the model
    # assigns to `false_negative_penalty_class` whenever the true label is any
    # *other* class -- i.e. specifically discourages "no tumor" whenever there
    # really is one, regardless of which tumor type. 0.0 disables it.
    false_negative_penalty_class: str = "notumor"
    false_negative_penalty_weight: float = 0.5
    early_stopping_patience: int = 5
    checkpoint_dir: Path = Path("checkpoints")
    history_dir: Path = Path("outputs")
    # ImageNet backbone: False = random init (slow); True = pretrained or local file / hub cache
    pretrained: bool = True
    imagenet_weights_path: Path | None = None

    @property
    def checkpoint_path(self) -> Path:
        """Per-architecture: training 3 models never overwrites each other's checkpoint."""
        return self.checkpoint_dir / f"{self.architecture}_best_model.pth"

    @property
    def history_path(self) -> Path:
        """Per-architecture: training 3 models never overwrites each other's history CSV."""
        return self.history_dir / f"{self.architecture}_metrics_history.csv"


In [ ]:
%%writefile src/data.py
from __future__ import annotations

import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict

import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from src.config import DataConfig


IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


@dataclass
class MRIDataBundle:
    train_loader: DataLoader
    val_loader: DataLoader
    test_loader: DataLoader
    class_to_idx: Dict[str, int]
    idx_to_class: Dict[int, str]
    train_size: int
    val_size: int
    test_size: int


def get_train_transforms(image_size: int) -> transforms.Compose:
    # RandomResizedCrop (instead of a fixed Resize) plus a bit more rotation/shift
    # than before: benchmarking showed glioma generalizing far worse than the other
    # 3 classes (test recall ~0.76-0.84 vs 0.94-1.0) despite ~98% val accuracy --
    # a classic overfit-to-Training-distribution signature, not a labeling bug (see
    # kb mistake/lesson from this session). More scale/position/rotation variation
    # forces the model to learn shape rather than the exact framing of the Training set.
    return transforms.Compose(
        [
            transforms.RandomResizedCrop(image_size, scale=(0.85, 1.0), ratio=(0.95, 1.05)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=20),
            transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
            transforms.ColorJitter(brightness=0.15, contrast=0.15),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ]
    )


def get_eval_transforms(image_size: int) -> transforms.Compose:
    return transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ]
    )


def _split_train_val_indices(num_items: int, val_split: float, seed: int) -> tuple[list[int], list[int]]:
    indices = list(range(num_items))
    rng = random.Random(seed)
    rng.shuffle(indices)
    val_count = int(num_items * val_split)
    val_indices = indices[:val_count]
    train_indices = indices[val_count:]
    return train_indices, val_indices


def build_dataloaders(config: DataConfig) -> MRIDataBundle:
    train_dir = config.data_root / config.train_dir_name
    test_dir = config.data_root / config.test_dir_name
    if not train_dir.exists() or not test_dir.exists():
        raise FileNotFoundError(
            f"Expected dataset folders at '{train_dir}' and '{test_dir}'. "
            "Run download_mri_dataset.py first."
        )

    full_train_for_aug = datasets.ImageFolder(train_dir, transform=get_train_transforms(config.image_size))
    full_train_for_eval = datasets.ImageFolder(train_dir, transform=get_eval_transforms(config.image_size))
    test_dataset = datasets.ImageFolder(test_dir, transform=get_eval_transforms(config.image_size))

    train_indices, val_indices = _split_train_val_indices(
        num_items=len(full_train_for_aug),
        val_split=config.val_split,
        seed=config.seed,
    )

    train_subset = Subset(full_train_for_aug, train_indices)
    val_subset = Subset(full_train_for_eval, val_indices)

    train_loader = DataLoader(
        train_subset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=config.num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_subset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    class_to_idx = full_train_for_aug.class_to_idx
    idx_to_class = {v: k for k, v in class_to_idx.items()}

    return MRIDataBundle(
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        class_to_idx=class_to_idx,
        idx_to_class=idx_to_class,
        train_size=len(train_subset),
        val_size=len(val_subset),
        test_size=len(test_dataset),
    )


def show_random_batch(data_bundle: MRIDataBundle, output_path: Path | None = None, max_images: int = 12) -> None:
    images, labels = next(iter(data_bundle.train_loader))
    images = images[:max_images]
    labels = labels[:max_images]

    cols = 4
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(12, 3 * rows))
    axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

    for i, ax in enumerate(axes):
        if i >= len(images):
            ax.axis("off")
            continue
        img = images[i].permute(1, 2, 0).cpu().numpy()
        img = (img * IMAGENET_STD) + IMAGENET_MEAN
        img = img.clip(0, 1)
        label_name = data_bundle.idx_to_class[int(labels[i].item())]
        ax.imshow(img)
        ax.set_title(label_name)
        ax.axis("off")

    plt.tight_layout()
    if output_path:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(output_path, dpi=150)
    else:
        plt.show()
    plt.close(fig)


if __name__ == "__main__":
    cfg = DataConfig()
    bundle = build_dataloaders(cfg)
    print("Class mapping:", bundle.class_to_idx)
    print(
        f"Sizes -> train: {bundle.train_size}, val: {bundle.val_size}, test: {bundle.test_size}"
    )
    show_random_batch(bundle, output_path=Path("outputs") / "sample_train_batch.png")
    print("Saved sample batch image to outputs/sample_train_batch.png")


In [ ]:
%%writefile src/train.py
from __future__ import annotations

import argparse
import csv
from dataclasses import asdict
from pathlib import Path

import torch
import torch.nn as nn
from torch.optim import Adam
from tqdm import tqdm

from src.config import DataConfig, TrainConfig
from src.data import build_dataloaders
from src.model import ARCHITECTURES, build_model, get_device


def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def run_epoch(
    model: torch.nn.Module,
    loader: torch.utils.data.DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer | None,
    device: torch.device,
    false_negative_penalty: tuple[int, float] | None = None,
) -> tuple[float, float]:
    """`false_negative_penalty`, if given, is (class_idx, weight): adds
    weight * mean(softmax_prob[:, class_idx]) over rows whose true label is NOT
    class_idx -- i.e. penalizes the model for leaning towards e.g. "notumor"
    on rows that are actually some other (tumor) class, on top of `criterion`.
    Applied in both train and eval so early stopping's val_loss reflects the
    same objective actually being optimized.
    """
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    pbar = tqdm(loader, leave=False)
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        with torch.set_grad_enabled(is_train):
            logits = model(images)
            loss = criterion(logits, labels)
            if false_negative_penalty is not None:
                penalty_idx, penalty_weight = false_negative_penalty
                if penalty_weight > 0:
                    is_other_class = labels != penalty_idx
                    if is_other_class.any():
                        probs = torch.softmax(logits, dim=1)
                        loss = loss + penalty_weight * probs[is_other_class, penalty_idx].mean()
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        preds = torch.argmax(logits, dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)
        total_loss += loss.item() * labels.size(0)

    avg_loss = total_loss / max(1, total_samples)
    avg_acc = total_correct / max(1, total_samples)
    return avg_loss, avg_acc


def save_history(history: list[dict], output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if not history:
        return
    keys = list(history[0].keys())
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(history)


def train(data_cfg: DataConfig, train_cfg: TrainConfig) -> Path:
    set_seed(data_cfg.seed)
    bundle = build_dataloaders(data_cfg)
    device = get_device()

    model = build_model(
        train_cfg.architecture,
        num_classes=len(bundle.class_to_idx),
        pretrained=train_cfg.pretrained,
        imagenet_weights_path=train_cfg.imagenet_weights_path,
    ).to(device)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = Adam(trainable_params, lr=train_cfg.learning_rate, weight_decay=train_cfg.weight_decay)

    # Per-class weight tensor, resolved by name against this dataset's actual
    # class_to_idx (never assume alphabetical order lines up with a hardcoded list).
    class_weights = torch.ones(len(bundle.class_to_idx))
    for class_name, weight_value in train_cfg.class_loss_weights.items():
        if class_name in bundle.class_to_idx:
            class_weights[bundle.class_to_idx[class_name]] = weight_value
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=train_cfg.label_smoothing)

    false_negative_penalty = None
    penalty_class_idx = bundle.class_to_idx.get(train_cfg.false_negative_penalty_class)
    if penalty_class_idx is not None and train_cfg.false_negative_penalty_weight > 0:
        false_negative_penalty = (penalty_class_idx, train_cfg.false_negative_penalty_weight)

    train_cfg.checkpoint_dir.mkdir(parents=True, exist_ok=True)
    best_ckpt_path = train_cfg.checkpoint_path

    best_val_loss = float("inf")
    patience_counter = 0
    history: list[dict] = []

    for epoch in range(1, train_cfg.epochs + 1):
        train_loss, train_acc = run_epoch(
            model, bundle.train_loader, criterion, optimizer, device, false_negative_penalty
        )
        val_loss, val_acc = run_epoch(
            model, bundle.val_loader, criterion, None, device, false_negative_penalty
        )

        epoch_row = {
            "epoch": epoch,
            "train_loss": round(train_loss, 6),
            "train_accuracy": round(train_acc, 6),
            "val_loss": round(val_loss, 6),
            "val_accuracy": round(val_acc, 6),
        }
        history.append(epoch_row)
        print(epoch_row)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(
                {
                    "architecture": train_cfg.architecture,
                    "model_state_dict": model.state_dict(),
                    "class_to_idx": bundle.class_to_idx,
                    "data_config": asdict(data_cfg),
                    "train_config": asdict(train_cfg),
                },
                best_ckpt_path,
            )
        else:
            patience_counter += 1

        if patience_counter >= train_cfg.early_stopping_patience:
            print(
                f"Early stopping at epoch {epoch} (no val loss improvement for "
                f"{train_cfg.early_stopping_patience} epochs)."
            )
            break

    save_history(history, train_cfg.history_path)
    print(f"Saved training history to {train_cfg.history_path}")
    print(f"Best model checkpoint: {best_ckpt_path}")
    return best_ckpt_path


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Train MRI classifier.")
    parser.add_argument("--architecture", choices=ARCHITECTURES, default="resnet50")
    parser.add_argument("--epochs", type=int, default=15)
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--num-workers", type=int, default=2)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument(
        "--imagenet-weights",
        type=str,
        default=None,
        help="Path to a local ImageNet weights .pth file (if download.pytorch.org is blocked).",
    )
    parser.add_argument(
        "--no-pretrained",
        action="store_true",
        help="Train from random init (no ImageNet weights; not recommended).",
    )
    return parser.parse_args()


if __name__ == "__main__":
    args = parse_args()
    data_config = DataConfig(batch_size=args.batch_size, num_workers=args.num_workers)
    imagenet_path = Path(args.imagenet_weights) if args.imagenet_weights else None
    train_config = TrainConfig(
        architecture=args.architecture,
        epochs=args.epochs,
        learning_rate=args.lr,
        pretrained=not args.no_pretrained,
        imagenet_weights_path=imagenet_path,
    )
    train(data_config, train_config)


### Install dependencies
Colab already ships a CUDA-enabled torch/torchvision -- skip the repo's CPU-era pins and install everything else.

In [ ]:
!grep -v -E "^(torch|torchvision)==" requirements.txt > /tmp/req_colab.txt
!pip install -q -r /tmp/req_colab.txt


### Kaggle credentials
Uses a `kaggle.json` dropped into `/content/` (drag it in via the folder icon in the
left sidebar -- kaggle.com -> Settings -> API -> Create New Token downloads this file).
Not the copyable `KGAT_...` token string -- that goes through a Kaggle-side introspection
endpoint that 404s on the currently published `kaggle` package, confirmed by real testing,
so this notebook doesn't use it at all. No interactive prompt here, so it doesn't block
Run All; if the file isn't there yet, raises a clear error instead of silently continuing
without credentials.


In [ ]:
import os, shutil
from pathlib import Path

os.environ.pop("KAGGLE_API_TOKEN", None)
kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)

json_candidates = ["/content/kaggle.json", "kaggle.json"]
src = next((c for c in json_candidates if os.path.exists(c)), None)
if src is None:
    raise FileNotFoundError(
        "No kaggle.json found. Click the folder icon in the left sidebar, drag your "
        "kaggle.json into /content/, then Runtime > Run all again."
    )
shutil.copy(src, kaggle_dir / "kaggle.json")
os.chmod(kaggle_dir / "kaggle.json", 0o600)
print("Kaggle credentials installed from kaggle.json.")


In [ ]:
!python download_mri_dataset.py


### Train all 3 architectures
Each uses the same fix: `RandomResizedCrop`+rotation+translate augmentation, glioma loss-weighted 1.5x, no label smoothing, plus a direct penalty on the model assigning "notumor" probability to rows that are actually some tumor class.

In [ ]:
!python -m src.train_all


### Benchmark against the real, never-trained-on `data/Testing` set

In [ ]:
!python -m src.benchmark


### Inspect glioma's numbers directly (the thing we're actually trying to fix)

In [ ]:
import json

for arch in ["resnet50", "efficientnet_b0", "vgg16"]:
    report = json.load(open(f"outputs/benchmark/{arch}/classification_report.json"))
    g = report["glioma"]
    print(
        arch.ljust(16),
        "glioma recall=%.4f" % g["recall"],
        " precision=%.4f" % g["precision"],
        " overall_acc=%.4f" % report["accuracy"],
    )


### Confusion matrices
The number that actually matters here is the glioma row's notumor column -- a real tumor read as "all clear". Watch that cell specifically across all 3.

In [ ]:
from IPython.display import Image, display

for arch in ["resnet50", "efficientnet_b0", "vgg16"]:
    print(arch)
    display(Image(filename=f"outputs/benchmark/{arch}/confusion_matrix.png"))


### Download the trained checkpoints + benchmark report back to your machine

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/mriscanner_results", "zip", ".", "checkpoints")
shutil.make_archive("/content/mriscanner_outputs", "zip", ".", "outputs")
files.download("/content/mriscanner_results.zip")
files.download("/content/mriscanner_outputs.zip")
